# Adaptation Results Analysis + Judge Audit

This notebook does two things in sequence:

**Part 1 — Results Analysis**
- Per-generator summaries: R1 pass rate, final pass rate, recovery by feedback
- Iteration breakdown (how many passed at round 1, 2, 3)
- Cross-generator comparison table
- Think trace inspection for failed rows

**Part 2 — Gemini Judge Audit**
- Re-parses `judge_failed` rows using Gemini to check structure validity,
  whether the judge reached a real decision, and whether failures were
  formatting artifacts vs genuine equivalence issues
- Merges audit columns back onto the original data (original columns preserved)
- Produces a final combined analysis with corrected outcome counts
- Exports a clean list for benchmark pooling

## 1. Install & Imports

In [ ]:
!pip install -q google-genai pandas openpyxl tqdm

In [ ]:
import re
import os
import json
import time
import glob
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from google import genai
from google.genai import types

## 2. Config

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
# Point to the directory containing all results_*.csv files
RESULTS_DIR = '.'
WORKDIR     = './outputs'
os.makedirs(WORKDIR, exist_ok=True)

AUDIT_OUT    = os.path.join(WORKDIR, 'judge_audit.csv')
COMBINED_OUT = os.path.join(WORKDIR, 'results_combined_audited.csv')
CLEAN_OUT    = os.path.join(WORKDIR, 'audit_clean_for_pool.csv')

# ── Gemini API (for audit) ────────────────────────────────────────────────────
GEMINI_API_KEY = 'REDACTED_USE_ENVIRONMENT_VARIABLE'
MODEL_ID       = 'gemini-2.5-flash'

# ── Audit config ──────────────────────────────────────────────────────────────
SAVE_EVERY        = 10
MAX_OUTPUT_TOKENS = 1024
THINKING_BUDGET   = 0      # pattern recognition — no thinking needed

# Which final_outcome values to send to the audit
# 'judge_failed' = judge parse exhausted, no verdict reached
# 'not_equivalent' = judge reached a verdict but failed — audit checks if
#                    it was only formatting errors that could be rescued
AUDIT_OUTCOMES = ['judge_failed', 'not_equivalent']

---
# Part 1 — Results Analysis

## 3. Load All Results

In [ ]:
files = sorted(glob.glob(os.path.join(RESULTS_DIR, 'results_*.csv')))
print(f'Found {len(files)} result files:')
for f in files:
    print(f'  {os.path.basename(f)}')

all_dfs = []
for f in files:
    d = pd.read_csv(f)
    all_dfs.append(d)

# Combined across all generators
pool = pd.concat(all_dfs, ignore_index=True)
print(f'\nTotal rows across all runs: {len(pool)}')
print(pool['generator_model'].value_counts().to_string())

In [ ]:
pool.to_excel("final_results.xlsx", index=False)

## 4. Per-Generator Summary

In [ ]:
def summarise_run(d: pd.DataFrame, label: str = ''):
    gen      = d['generator_model'].iloc[0]
    n        = len(d)
    r1_pass  = d['r1_equivalence'].eq(True).mean()
    fin_pass = d['final_equivalence'].eq(True).mean()
    recovered = ((d['r1_equivalence'] != True) &
                 (d['final_equivalence'] == True)).sum()

    outcome_counts   = d['final_outcome'].value_counts() if 'final_outcome' in d.columns else {}
    n_equivalent     = outcome_counts.get('equivalent',     0)
    n_not_equivalent = outcome_counts.get('not_equivalent', 0)
    n_judge_failed   = outcome_counts.get('judge_failed',   0)
    n_gen_failed     = outcome_counts.get('gen_failed',     0)

    # Iteration breakdown: how many passed at each round
    r1_pass_n  = (d['r1_equivalence'] == True).sum()
    r2_pass_n  = ((d['r1_equivalence'] != True) &
                  (d['final_equivalence'] == True) &
                  (d['iterations_used'] == 2)).sum()
    r3_pass_n  = ((d['r1_equivalence'] != True) &
                  (d['final_equivalence'] == True) &
                  (d['iterations_used'] >= 3)).sum()

    judge_col = d['judge_model'].iloc[0] if 'judge_model' in d.columns else 'unknown'
    backend   = d['judge_backend'].iloc[0] if 'judge_backend' in d.columns else 'unknown'
    gen_think = d['generator_thinking'].iloc[0] if 'generator_thinking' in d.columns else '?'

    print(f"\n{'='*60}")
    if label:
        print(f'Run       : {label}')
    print(f'Generator : {gen}')
    if gen_think != '?':
        print(f'Thinking  : {"on" if gen_think else "off"}')
    print(f'Judge     : {judge_col} ({backend})')
    print(f'N         : {n}')
    print(f"{'─'*60}")
    print(f'Round 1 pass rate  (independent) : {r1_pass:.1%}  ({r1_pass_n}/{n})')
    print(f'Final pass rate    (dependent)   : {fin_pass:.1%}')
    print(f'  Passed at Round 1              : {r1_pass_n}')
    print(f'  Passed at Round 2 (recovered)  : {r2_pass_n}')
    print(f'  Passed at Round 3 (recovered)  : {r3_pass_n}')
    print(f'  Total recovered by feedback    : {recovered}')
    print(f"Mean iterations                  : {d['iterations_used'].mean():.2f}")
    print(f"{'─'*60}")
    print('Final outcome breakdown:')
    print(f'  equivalent     (proven pass)   : {n_equivalent}')
    print(f'  not_equivalent (proven fail)   : {n_not_equivalent}')
    print(f'  judge_failed   (inconclusive)  : {n_judge_failed}')
    if n_gen_failed:
        print(f'  gen_failed     (no output)     : {n_gen_failed}')
    print(f"{'─'*60}")
    print('Iteration distribution:')
    print(d['iterations_used'].value_counts().sort_index().to_string())
    print(f"{'─'*60}")
    print('Top domains (Round 1):')
    print(d['r1_target_domain'].value_counts().head(10).to_string())


for gen_name, g in pool.groupby('generator_model'):
    summarise_run(g, label=gen_name)

## 5. Cross-Generator Comparison

In [ ]:
def build_comparison(pool: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for gen_name, g in pool.groupby('generator_model'):
        outcome_counts = g['final_outcome'].value_counts() if 'final_outcome' in g.columns else {}
        gen_think = g['generator_thinking'].iloc[0] if 'generator_thinking' in g.columns else None

        r1_pass_n = (g['r1_equivalence'] == True).sum()
        r2_pass_n = ((g['r1_equivalence'] != True) &
                     (g['final_equivalence'] == True) &
                     (g['iterations_used'] == 2)).sum()
        r3_pass_n = ((g['r1_equivalence'] != True) &
                     (g['final_equivalence'] == True) &
                     (g['iterations_used'] >= 3)).sum()

        rows.append({
            'generator':      gen_name,
            'thinking':       gen_think,
            'n':              len(g),
            'r1_pass_%':      round(g['r1_equivalence'].eq(True).mean() * 100, 1),
            'final_pass_%':   round(g['final_equivalence'].eq(True).mean() * 100, 1),
            'pass_r1':        r1_pass_n,
            'pass_r2':        r2_pass_n,
            'pass_r3':        r3_pass_n,
            'mean_iters':     round(g['iterations_used'].mean(), 2),
            'equivalent':     outcome_counts.get('equivalent',     0),
            'not_equivalent': outcome_counts.get('not_equivalent', 0),
            'judge_failed':   outcome_counts.get('judge_failed',   0),
            'gen_failed':     outcome_counts.get('gen_failed',     0),
        })
    cmp = pd.DataFrame(rows).sort_values('final_pass_%', ascending=False)
    return cmp


cmp = build_comparison(pool)
print('\nCross-generator comparison:')
print(cmp.to_string(index=False))
cmp.to_csv(os.path.join(WORKDIR, 'comparison_all_generators.csv'), index=False)

## 6. Think Trace Inspection (failed rows)

In [ ]:
def inspect_think_traces(d: pd.DataFrame, n: int = 5):
    """Show judge and generator think traces for failed rows."""
    failed = d[d['final_equivalence'] != True].head(n)
    for _, row in failed.iterrows():
        print(f"\nP{row['Puzzle_Number']} Q{row['Question_Number']}  "
              f"[{row.get('generator_model','')}]")
        print(f"  Outcome      : {row.get('final_outcome', '')}")
        print(f"  Iterations   : {row['iterations_used']}")
        print(f"  Format check : {row['r1_format_check']}")
        print(f"  Feedback     : {row['r1_feedback']}")
        if 'final_gen_think_trace' in row and str(row['final_gen_think_trace']).strip():
            print(f'  Gen think trace (first 400 chars):')
            print(f"    {str(row['final_gen_think_trace'])[:400]}")
        print(f'  Judge think trace (first 600 chars):')
        print(f"    {str(row['final_judge_think_trace'])[:600]}")


# Run on the combined pool or a single generator's df
inspect_think_traces(pool, n=5)
# inspect_think_traces(all_dfs[0], n=3)

---
# Part 2 — Gemini Judge Audit

## 7. Init Gemini Client

In [ ]:
client = genai.Client(api_key=GEMINI_API_KEY)
print(f'Client ready. Model: {MODEL_ID}')

## 8. Audit Helpers

In [ ]:
def save_results(records: list, path: str):
    pd.DataFrame(records).to_csv(path, index=False)


def load_completed(path: str, key_cols: list):
    if not os.path.exists(path):
        return [], set()
    done_df   = pd.read_csv(path)
    done_keys = set(zip(*[done_df[c] for c in key_cols]))
    records   = done_df.to_dict('records')
    print(f'  Resuming: {len(records)} rows already in {os.path.basename(path)}')
    return records, done_keys


def generate_audit(system_prompt: str, user_prompt: str) -> tuple:
    """
    Returns (raw_text, elapsed_minutes).
    Retries on 503/429 with exponential backoff starting at 30s.
    """
    max_retries = 5
    wait = 30

    for attempt in range(max_retries):
        try:
            t0 = time.time()
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_prompt,
                    max_output_tokens=MAX_OUTPUT_TOKENS,
                    temperature=0.0,
                    thinking_config=types.ThinkingConfig(
                        thinking_budget=THINKING_BUDGET,
                    ),
                ),
            )
            elapsed = (time.time() - t0) / 60.0
            raw = ''
            for part in response.candidates[0].content.parts:
                if not getattr(part, 'thought', False):
                    raw += (part.text or '')
            return raw.strip(), elapsed

        except Exception as e:
            if attempt < max_retries - 1 and ('503' in str(e) or '429' in str(e)):
                print(f'  [Retry {attempt+1}/{max_retries}] {e} — waiting {wait}s')
                time.sleep(wait)
                wait *= 2
            else:
                raise


def parse_audit_response(raw: str) -> dict:
    try:
        clean = re.sub(r'```(?:json)?', '', raw).strip().rstrip('`').strip()
        match = re.search(r'\{.*\}', clean, re.DOTALL)
        if match:
            return json.loads(match.group(0))
    except Exception:
        pass
    return {'parse_error': True, 'raw_audit_response': raw[:500]}

## 9. Audit System Prompt

In [ ]:
AUDIT_SYSTEM_PROMPT = """\
You are auditing the output of an automated puzzle equivalence judge.
You will receive:
  1. An adapted puzzle
  2. The judge's verdict (JSON)
  3. The judge's think trace (may be empty)

Your job is to check four things and report only what you observe.
Do NOT re-judge equivalence yourself. Do NOT solve the puzzle.

---
CHECK 1 — PUZZLE STRUCTURE
Does the adapted puzzle contain all of the following?
  - <context> block
  - <clues> block
  - <question> block
  - Answer choices A through E

Set puzzle_structure_valid = true only if ALL are present.
Otherwise list what is missing in structure_issues.

---
CHECK 2 — JUDGE REACHED A DECISION
Look at the judge's think trace and verdict.
Did the judge actually reach a final logical_equivalence_confirmed value
(true or false), or did it fail before concluding (e.g. parse error,
empty verdict, judge_parse_exhausted)?

Set judge_reached_decision = true if a real true/false was produced.
Set judge_reached_decision = false if the verdict is an error or inconclusive.

---
CHECK 3 — NATURE OF JUDGE FAILURE (only if judge said FAIL)
If the judge returned format_check = FAIL, look at the feedback field.
Classify the failure:
  - "formatting_only"    : failures are purely structural (missing tags,
                           wrong choice count, leaked entity names)
  - "equivalence_issue"  : judge cited genuine logical differences
                           (wrong clue type, direction flipped, clue merged)
  - "mixed"              : both formatting and equivalence issues cited
  - "inconclusive"       : judge did not reach a decision

Set failure_nature to one of the four values above.

---
CHECK 4 — JUDGE'S FINAL DECISION
Report what logical_equivalence_confirmed value the judge produced.
  - true   : judge confirmed equivalent
  - false  : judge rejected
  - null   : judge did not reach a decision

---
OUTPUT JSON ONLY. No markdown, no explanation, no ```json blocks.

{
  "puzzle_structure_valid": true|false,
  "structure_issues": "<comma-separated list of missing elements or empty string>",
  "judge_reached_decision": true|false,
  "failure_nature": "formatting_only|equivalence_issue|mixed|inconclusive",
  "judge_final_decision": true|false|null,
  "audit_notes": "<one sentence summary of any notable issue, or empty string>"
}
"""

## 10. Run Audit

In [ ]:
# Select rows to audit from the combined pool
audit_input = pool[pool['final_outcome'].isin(AUDIT_OUTCOMES)].copy()
print(f'Rows to audit: {len(audit_input)}')
print(audit_input['final_outcome'].value_counts().to_string())

KEY_COLS = ['Puzzle_Number', 'Question_Number', 'generator_model']
audit_records, audit_done = load_completed(AUDIT_OUT, KEY_COLS)

for _, row in tqdm(audit_input.iterrows(), total=len(audit_input), desc='Audit'):
    key = (row['Puzzle_Number'], row['Question_Number'], row['generator_model'])
    if key in audit_done:
        continue

    user_prompt = f"""<adapted_puzzle>
{row['final_adapted_puzzle']}
</adapted_puzzle>

<judge_verdict>
{row['final_judge_verdict']}
</judge_verdict>

<judge_think_trace>
{row.get('final_judge_think_trace') or '(none)'}
</judge_think_trace>"""

    raw_response = ''
    elapsed      = 0.0
    audit        = {}

    try:
        raw_response, elapsed = generate_audit(AUDIT_SYSTEM_PROMPT, user_prompt)
        audit = parse_audit_response(raw_response)
    except Exception as e:
        print(f'  [P{row["Puzzle_Number"]} Q{row["Question_Number"]} {row["generator_model"]}] Error: {e}')
        raw_response = f'ERROR: {e}'
        audit = {'parse_error': True}

    audit_records.append({
        'Puzzle_Number':          row['Puzzle_Number'],
        'Question_Number':        row['Question_Number'],
        'generator_model':        row['generator_model'],
        'original_final_outcome': row['final_outcome'],
        # Audit fields
        'puzzle_structure_valid': audit.get('puzzle_structure_valid'),
        'structure_issues':       audit.get('structure_issues', ''),
        'judge_reached_decision': audit.get('judge_reached_decision'),
        'failure_nature':         audit.get('failure_nature', ''),
        'judge_final_decision':   audit.get('judge_final_decision'),
        'audit_notes':            audit.get('audit_notes', ''),
        'audit_parse_error':      audit.get('parse_error', False),
        'raw_audit_response':     raw_response,
        'audit_elapsed_min':      round(elapsed, 3),
    })
    audit_done.add(key)

    if len(audit_records) % SAVE_EVERY == 0:
        save_results(audit_records, AUDIT_OUT)
        print(f'  [Checkpoint] {len(audit_records)} rows saved.')

save_results(audit_records, AUDIT_OUT)
audit_df = pd.DataFrame(audit_records)
print(f'\nAudit complete: {len(audit_df)} rows -> {AUDIT_OUT}')

## 11. Merge Audit Back onto Original Data

In [ ]:
audit_df = pd.read_csv(AUDIT_OUT)

# Merge audit columns onto the full pool — original columns all preserved
# Only audited rows will have audit values; equivalent rows get NaN (expected)
audit_cols = [
    'Puzzle_Number', 'Question_Number', 'generator_model',
    'puzzle_structure_valid', 'structure_issues',
    'judge_reached_decision', 'failure_nature',
    'judge_final_decision', 'audit_notes',
    'audit_parse_error', 'raw_audit_response', 'audit_elapsed_min',
]
combined = pool.merge(
    audit_df[audit_cols],
    on=['Puzzle_Number', 'Question_Number', 'generator_model'],
    how='left',
)

# Derive corrected_outcome:
# - equivalent rows stay equivalent
# - judge_failed/not_equivalent rows where audit says structure valid +
#   formatting_only failure → mark as 'rescued_formatting'
# - judge_failed rows where judge never reached a decision → 'judge_inconclusive'
# - rest stay as original outcome
def derive_corrected_outcome(row):
    if row['final_outcome'] == 'equivalent':
        return 'equivalent'
    if row['final_outcome'] == 'gen_failed':
        return 'gen_failed'
    if pd.isna(row.get('puzzle_structure_valid')):
        return row['final_outcome']
    if row['puzzle_structure_valid'] == True and row['failure_nature'] == 'formatting_only':
        return 'rescued_formatting'
    if row['failure_nature'] == 'inconclusive' or row['judge_reached_decision'] == False:
        return 'judge_inconclusive'
    return row['final_outcome']

combined['corrected_outcome'] = combined.apply(derive_corrected_outcome, axis=1)

combined.to_csv(COMBINED_OUT, index=False)
print(f'Combined dataset: {len(combined)} rows -> {COMBINED_OUT}')

orig    = combined['final_outcome'].value_counts()
revised = combined['corrected_outcome'].value_counts()

comparison = pd.DataFrame({
    'original':   orig,
    'post_audit': revised,
}).fillna(0).astype(int)
comparison['delta'] = comparison['post_audit'] - comparison['original']

print('\nOriginal vs Post-Audit Outcome Breakdown:')
print(comparison.to_string())

## 12. Post-Audit Analysis

In [ ]:
print(f"{'='*60}")
print('POST-AUDIT RESULTS SUMMARY')
print(f"{'='*60}")

for gen, g in combined.groupby('generator_model'):
    n          = len(g)
    equivalent = (g['corrected_outcome'] == 'equivalent').sum()
    rescued    = (g['corrected_outcome'] == 'rescued_formatting').sum()
    total_pass = equivalent + rescued
    not_equiv  = (g['corrected_outcome'] == 'not_equivalent').sum()
    inconc     = (g['corrected_outcome'] == 'judge_inconclusive').sum()
    gen_fail   = (g['corrected_outcome'] == 'gen_failed').sum()

    print(f'\n{gen}')
    print(f"  N                         : {n}")
    print(f"  Equivalent (judge pass)   : {equivalent}")
    print(f"  Rescued (formatting only) : {rescued}")
    print(f"  Total passable            : {total_pass}  ({total_pass/n:.1%})")
    print(f"  Not equivalent            : {not_equiv}")
    print(f"  Judge inconclusive        : {inconc}")
    if gen_fail:
        print(f"  Gen failed                : {gen_fail}")

print(f"\n{'─'*60}")
print('Audit failure_nature breakdown (across all generators):')
print(audit_df['failure_nature'].value_counts().to_string())
print()
print('Structure issues breakdown:')
invalid = audit_df[audit_df['puzzle_structure_valid'] == False]
print(f'  Structure invalid rows: {len(invalid)}')
if len(invalid):
    print(invalid['structure_issues'].value_counts().head(10).to_string())

## 13. Export Clean List for Benchmark Pooling

In [ ]:
# Rows safe for the benchmark pool:
# - equivalent (judge confirmed)
# - rescued_formatting (structure valid, only formatting issues)
clean = combined[
    combined['corrected_outcome'].isin(['equivalent', 'rescued_formatting'])
][[
    'Puzzle_Number', 'Question_Number', 'generator_model',
    'final_outcome', 'corrected_outcome',
    'failure_nature', 'audit_notes',
    'final_adapted_puzzle', 'final_target_domain',
    'Answer',
]].copy()

clean.to_csv(CLEAN_OUT, index=False)
print(f'Clean rows for benchmark pooling: {len(clean)} -> {CLEAN_OUT}')
print('\nBreakdown by generator:')
print(clean.groupby('generator_model')['corrected_outcome'].value_counts().to_string())
clean.head(10)

In [ ]:
def summarise_run_combined(d: pd.DataFrame, label: str = ''):
    gen      = d['generator_model'].iloc[0]
    n        = len(d)

    outcome_counts    = d['corrected_outcome'].value_counts()
    n_equivalent      = outcome_counts.get('equivalent',          0)
    n_rescued         = outcome_counts.get('rescued_formatting',  0)
    n_not_equivalent  = outcome_counts.get('not_equivalent',      0)
    n_inconclusive    = outcome_counts.get('judge_inconclusive',  0)
    n_gen_failed      = outcome_counts.get('gen_failed',          0)
    total_pass        = n_equivalent + n_rescued

    print(f"\n{'='*60}")
    if label:
        print(f'Run       : {label}')
    print(f'Generator : {gen}')
    print(f'N         : {n}')
    print(f"{'─'*60}")
    print(f'Post-audit outcome breakdown:')
    print(f'  equivalent         (judge pass)       : {n_equivalent}')
    print(f'  rescued_formatting (formatting only)  : {n_rescued}')
    print(f'  ─────────────────────────────────────')
    print(f'  Total passable                        : {total_pass}  ({total_pass/n:.1%})')
    print(f'  not_equivalent     (genuine fail)     : {n_not_equivalent}')
    print(f'  judge_inconclusive (no decision)      : {n_inconclusive}')
    if n_gen_failed:
        print(f'  gen_failed         (no output)        : {n_gen_failed}')
    print(f"{'─'*60}")
    print('Failure nature breakdown (audited rows):')
    audited = d[d['failure_nature'].notna()]
    if len(audited):
        print(audited['failure_nature'].value_counts().to_string())


for gen_name, g in combined.groupby('generator_model'):
    summarise_run_combined(g, label=gen_name)

In [ ]:
def build_comparison_combined(combined: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for gen_name, g in combined.groupby('generator_model'):
        n              = len(g)
        outcome_counts = g['corrected_outcome'].value_counts()
        n_equivalent   = outcome_counts.get('equivalent',         0)
        n_rescued      = outcome_counts.get('rescued_formatting', 0)
        total_pass     = n_equivalent + n_rescued

        r1_pass_n = (g['r1_equivalence'] == True).sum()
        r2_pass_n = ((g['r1_equivalence'] != True) &
                     (g['final_equivalence'] == True) &
                     (g['iterations_used'] == 2)).sum()
        r3_pass_n = ((g['r1_equivalence'] != True) &
                     (g['final_equivalence'] == True) &
                     (g['iterations_used'] >= 3)).sum()

        rows.append({
            'generator':          gen_name,
            'n':                  n,
            'r1_pass_%':          round(g['r1_equivalence'].eq(True).mean() * 100, 1),
            'pass_r1':            r1_pass_n,
            'pass_r2':            r2_pass_n,
            'pass_r3':            r3_pass_n,
            'mean_iters':         round(g['iterations_used'].mean(), 2),
            'equivalent':         n_equivalent,
            'rescued_formatting': n_rescued,
            'total_pass':         total_pass,
            'total_pass_%':       round(total_pass / n * 100, 1),
            'not_equivalent':     outcome_counts.get('not_equivalent',     0),
            'judge_inconclusive': outcome_counts.get('judge_inconclusive', 0),
            'gen_failed':         outcome_counts.get('gen_failed',         0),
        })

    cmp = pd.DataFrame(rows).sort_values('total_pass_%', ascending=False)
    return cmp


cmp_combined = build_comparison_combined(combined)
print('\nPost-audit cross-generator comparison:')
print(cmp_combined.to_string(index=False))
cmp_combined.to_csv(os.path.join(WORKDIR, 'comparison_post_audit.csv'), index=False)

# Judge Failure Summary

In [ ]:
# Collect all non-empty audit notes
notes = combined[combined['audit_notes'].notna() & (combined['audit_notes'] != '')]

notes_by_gen = {}
for gen_name, g in notes.groupby('generator_model'):
    notes_by_gen[gen_name] = g['audit_notes'].tolist()

# Build the prompt
notes_text = ''
for gen_name, note_list in notes_by_gen.items():
    notes_text += f'\n\n=== {gen_name} ===\n'
    for i, note in enumerate(note_list, 1):
        notes_text += f'{i}. {note}\n'

user_prompt = f"""Below are audit notes from a judge audit across {len(notes)} adapted logic puzzles,
grouped by generator model. Each note describes a notable issue found in either the
adapted puzzle structure or the judge's evaluation.

{notes_text}

Please provide:
1. A concise summary of the most common failure patterns across all generators
2. Per-generator patterns — what does each model tend to struggle with specifically
3. Any surprising or unexpected findings
4. Which failure types suggest a fixable prompt issue vs a fundamental model limitation"""

system_prompt = """You are analyzing audit notes from an automated puzzle adaptation pipeline.
Be concise, specific, and data-driven. Group similar issues together.
Where counts matter, estimate from the notes provided."""

raw, elapsed = generate_audit(system_prompt, user_prompt)
print(raw)
print(f'\n(elapsed: {elapsed:.2f} min)')

# Sample

In [ ]:
import pandas as pd

# Load original dataset (has Solved_Percentile) and clean pool
original = pd.read_excel('/content/puzzle_samples.xlsx')  # or .csv
clean    = pd.read_csv('/content/audit_clean_for_pool.csv')

# Only equivalent rows (judge confirmed pass)
equiv = clean[clean['corrected_outcome'] == 'equivalent'].copy()

# Merge in Solved_Percentile from original on Puzzle_Number + Question_Number
equiv = equiv.merge(
    original[['Puzzle_Number', 'Question_Number', 'Solved_Percentile']],
    on=['Puzzle_Number', 'Question_Number'],
    how='left'
)

# Map difficulty
def map_difficulty(p):
    if p < 30:
        return 'Hard'
    elif p < 60:
        return 'Medium'
    else:
        return 'Easy'

equiv['difficulty'] = equiv['Solved_Percentile'].apply(map_difficulty)

print('Equivalent rows by difficulty:')
print(equiv['difficulty'].value_counts().to_string())

# Stratified sample — 33 Hard, 33 Medium, 34 Easy = 100 total
sample = (
    equiv.groupby('difficulty', group_keys=False)
    .apply(lambda g: g.sample(min(len(g), 34), random_state=42))
)

# Trim to exactly 100 if rounding gives 101
sample = sample.sample(frac=1, random_state=42)

print(f'\nSampled: {len(sample)}')
print(sample['difficulty'].value_counts().to_string())


In [ ]:
sample.to_csv('benchmark_100.csv', index=False)
print('\nSaved -> benchmark_100.csv')

In [ ]:
sample.columns

In [ ]:
instruction = (
    "Your task is to provide the correct answer to the given puzzle using the clues."
)

sample["Sample"] = instruction + "\n" + sample['final_adapted_puzzle']

In [ ]:
sample.head()

In [ ]:
sample.to_excel("benchmark_100_formatted.xlsx",index=False)

---
# Part 3 — Adaptation Pipeline Visualizations

## Viz Setup — Imports & Generator Order

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches

# Use combined if audit is done, else pool
viz_df = combined if 'combined' in dir() else pool

OUTDIR = WORKDIR
os.makedirs(OUTDIR, exist_ok=True)

# Consistent generator order (sort by final pass rate descending)
_cmp = build_comparison(pool)
GEN_ORDER = _cmp['generator'].tolist()
N_GEN     = len(GEN_ORDER)
COLORS    = plt.cm.tab10(np.linspace(0, 0.9, N_GEN))
GEN_COLOR = dict(zip(GEN_ORDER, COLORS))


## Viz 1 — Adaptation Success Rate by Generator

In [ ]:
# ── 1. Adaptation Success Rate by Generator ───────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

cmp = build_comparison(pool).sort_values('final_pass_%', ascending=True)
colors = [GEN_COLOR[g] for g in cmp['generator']]
bars = ax.barh(cmp['generator'], cmp['final_pass_%'],
               color=colors, edgecolor='black', linewidth=0.5)
ax.bar_label(bars, fmt='%.1f%%', padding=4, fontsize=9)

ax.axvline(x=100, color='grey', linestyle='--', alpha=0.3)
ax.set_xlabel('Adaptation Pass Rate (%)', fontweight='bold')
ax.set_title('Adaptation Success Rate by Generator\n(Judge-confirmed equivalent)', fontweight='bold')
ax.set_xlim(0, 115)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'viz1_adaptation_success_rate.png'), dpi=150, bbox_inches='tight')
plt.show()


## Viz 2 — Iteration Distribution Heatmap

In [ ]:
# ── 2. Iteration Distribution Heatmap ────────────────────────────────────────
# % of total puzzles that passed at each round per generator
iter_data = []
for gen_name, g in pool.groupby('generator_model'):
    n = len(g)
    r1 = (g['r1_equivalence'] == True).sum() / n * 100
    r2 = ((g['r1_equivalence'] != True) & (g['final_equivalence'] == True) & (g['iterations_used'] == 2)).sum() / n * 100
    r3 = ((g['r1_equivalence'] != True) & (g['final_equivalence'] == True) & (g['iterations_used'] >= 3)).sum() / n * 100
    failed = 100 - r1 - r2 - r3
    iter_data.append({'generator': gen_name, 'Round 1': r1, 'Round 2': r2, 'Round 3': r3, 'Failed': failed})

iter_df = pd.DataFrame(iter_data).set_index('generator').reindex(GEN_ORDER)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(iter_df.values, cmap='RdYlGn', vmin=0, vmax=100, aspect='auto')
plt.colorbar(im, ax=ax, label='% of puzzles')

ax.set_xticks(range(len(iter_df.columns)))
ax.set_xticklabels(iter_df.columns, fontweight='bold')
ax.set_yticks(range(len(iter_df.index)))
ax.set_yticklabels(iter_df.index)

for i in range(len(iter_df.index)):
    for j in range(len(iter_df.columns)):
        val = iter_df.values[i, j]
        ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                fontsize=9, color='black' if val < 70 else 'white', fontweight='bold')

ax.set_title('Iteration Distribution by Generator\n(% of total puzzles passing at each round)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'viz2_iteration_heatmap.png'), dpi=150, bbox_inches='tight')
plt.show()


## Viz 3 — Feedback Loop Recovery Rate

In [ ]:
# ── 3. Recovery Rate — R1 fail → R2/R3 pass ──────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

recovery_data = []
for gen_name, g in pool.groupby('generator_model'):
    r1_fail = (g['r1_equivalence'] != True).sum()
    recovered = ((g['r1_equivalence'] != True) & (g['final_equivalence'] == True)).sum()
    recovery_rate = recovered / r1_fail * 100 if r1_fail > 0 else 0
    recovery_data.append({'generator': gen_name, 'r1_fail': r1_fail,
                           'recovered': recovered, 'recovery_rate': recovery_rate})

rec_df = pd.DataFrame(recovery_data).sort_values('recovery_rate', ascending=True)

colors = [GEN_COLOR[g] for g in rec_df['generator']]
bars = ax.barh(rec_df['generator'], rec_df['recovery_rate'],
               color=colors, edgecolor='black', linewidth=0.5)

for bar, (_, row) in zip(bars, rec_df.iterrows()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{row["recovered"]}/{row["r1_fail"]}',
            va='center', fontsize=9)

ax.set_xlabel('Recovery Rate (% of R1-failed puzzles rescued by feedback)', fontweight='bold')
ax.set_title('Feedback Loop Recovery Rate by Generator\n(fraction of R1 failures rescued at R2/R3)', fontweight='bold')
ax.set_xlim(0, 115)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'viz3_recovery_rate.png'), dpi=150, bbox_inches='tight')
plt.show()


## Viz 4 — Outcome Breakdown Stacked Bar

In [ ]:
# ── 4. Outcome Breakdown Stacked Bar ─────────────────────────────────────────
outcome_col = 'corrected_outcome' if 'corrected_outcome' in viz_df.columns else 'final_outcome'

outcome_map = {
    'equivalent':          ('#2ca02c', 'Equivalent'),
    'rescued_formatting':  ('#98df8a', 'Rescued (formatting)'),
    'not_equivalent':      ('#d62728', 'Not Equivalent'),
    'judge_failed':        ('#ff7f0e', 'Judge Failed'),
    'judge_inconclusive':  ('#ffbb78', 'Judge Inconclusive'),
    'gen_failed':          ('#7f7f7f', 'Gen Failed'),
}

stacked = {}
for outcome, (color, label) in outcome_map.items():
    stacked[outcome] = []
    for gen in GEN_ORDER:
        g = viz_df[viz_df['generator_model'] == gen]
        stacked[outcome].append((g[outcome_col] == outcome).sum() / len(g) * 100 if len(g) else 0)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(N_GEN)
bottom = np.zeros(N_GEN)

for outcome, (color, label) in outcome_map.items():
    vals = np.array(stacked[outcome])
    if vals.sum() > 0:
        ax.bar(x, vals, bottom=bottom, color=color, label=label,
               edgecolor='white', linewidth=0.5)
        for i, (v, b) in enumerate(zip(vals, bottom)):
            if v > 4:
                ax.text(i, b + v/2, f'{v:.0f}%', ha='center', va='center',
                        fontsize=8, color='white', fontweight='bold')
        bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(GEN_ORDER, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('% of puzzles', fontweight='bold')
ax.set_title(f'Outcome Breakdown by Generator\n(based on {outcome_col})', fontweight='bold')
ax.set_ylim(0, 105)
ax.legend(loc='upper right', fontsize=8, framealpha=0.9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'viz4_outcome_breakdown.png'), dpi=150, bbox_inches='tight')
plt.show()


## Viz 5 — R1 vs Final Pass Rate Slope Chart

In [ ]:
# ── 5. R1 vs Final Pass Rate — slope chart ────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

for gen_name, g in pool.groupby('generator_model'):
    r1   = g['r1_equivalence'].eq(True).mean() * 100
    fin  = g['final_equivalence'].eq(True).mean() * 100
    color = GEN_COLOR[gen_name]
    ax.plot([0, 1], [r1, fin], 'o-', color=color, linewidth=2, markersize=7)
    ax.text(-0.05, r1, f'{gen_name}', ha='right', va='center', fontsize=8, color=color)
    ax.text(1.05,  fin, f'{fin:.1f}%', ha='left',  va='center', fontsize=8, color=color)

ax.set_xticks([0, 1])
ax.set_xticklabels(['Round 1\n(Independent)', 'Final\n(After Feedback)'], fontweight='bold')
ax.set_ylabel('Pass Rate (%)', fontweight='bold')
ax.set_title('R1 vs Final Pass Rate\n(slope shows feedback loop improvement)', fontweight='bold')
ax.set_xlim(-0.4, 1.5)
ax.set_ylim(0, 105)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'viz5_r1_vs_final_slope.png'), dpi=150, bbox_inches='tight')
plt.show()


## Viz 6 — Top Adaptation Domains per Generator

In [ ]:
# ── 6. Top Adaptation Domains per Generator ───────────────────────────────────
TOP_N = 6
n_gens = N_GEN
ncols  = 3
nrows  = -(-n_gens // ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for idx, gen_name in enumerate(GEN_ORDER):
    ax = axes[idx]
    g  = pool[pool['generator_model'] == gen_name]

    top = g['r1_target_domain'].value_counts().head(TOP_N)
    colors = plt.cm.Pastel1(np.linspace(0, 0.8, len(top)))
    bars = ax.barh(top.index[::-1], top.values[::-1], color=colors[::-1],
                   edgecolor='black', linewidth=0.4)
    ax.bar_label(bars, padding=2, fontsize=8)
    ax.set_title(gen_name, fontweight='bold', fontsize=9)
    ax.set_xlabel('Count', fontsize=8)
    ax.set_xlim(0, top.values.max() * 1.25)
    ax.grid(axis='x', alpha=0.3)

for ax in axes[n_gens:]:
    ax.set_visible(False)

plt.suptitle(f'Top {TOP_N} Adaptation Domains per Generator (Round 1)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'viz6_top_domains.png'), dpi=150, bbox_inches='tight')
plt.show()


## Viz 7 — Judge Failure Rate by Generator

In [ ]:
# ── 7. Judge Failure Rate by Generator ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

fail_data = []
for gen_name, g in pool.groupby('generator_model'):
    n            = len(g)
    judge_failed = (g['final_outcome'] == 'judge_failed').sum()
    not_equiv    = (g['final_outcome'] == 'not_equivalent').sum()
    fail_data.append({
        'generator':    gen_name,
        'judge_failed': judge_failed / n * 100,
        'not_equiv':    not_equiv    / n * 100,
    })

fail_df = pd.DataFrame(fail_data).set_index('generator').reindex(GEN_ORDER)
x      = np.arange(N_GEN)
width  = 0.35

ax.bar(x - width/2, fail_df['judge_failed'], width, label='Judge Parse Failed',
       color='#ff7f0e', edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, fail_df['not_equiv'],    width, label='Not Equivalent',
       color='#d62728', edgecolor='black', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(GEN_ORDER, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('% of puzzles', fontweight='bold')
ax.set_title('Judge Failure Rate by Generator', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, 'viz7_judge_failure_rate.png'), dpi=150, bbox_inches='tight')
plt.show()
